# Tarea 11 · Pipeline DADA2

**Objetivo:** trabajar con 12 muestras (6 Control y 6 Crohn) cuando los metadatos originales lo permitan, procesar lecturas y guardar un objeto `phyloseq` para la siguiente clase.

**Instrucción:** ejecutar **solo una** de las opciones A o B. Los FASTQ deben ser archivos originales emparejados, no FASTQ previamente filtrados.


## 1. Preparación y librerías

In [ ]:
# Google Colab: seleccionar entorno R antes de ejecutar
if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
for (p in c("dada2", "phyloseq", "Biostrings", "ShortRead")) {
  if (!requireNamespace(p, quietly=TRUE)) BiocManager::install(p, ask=FALSE, update=FALSE)
}
library(dada2); library(phyloseq); library(Biostrings); library(ShortRead)
for (x in c('datos','filtrados','resultados','referencias')) dir.create(x,showWarnings=FALSE)
# Persistencia: montar Google Drive en Colab desde el panel Archivos > Montar Drive.
# Si no se monta, se guardará solo en la sesión temporal de Colab.
carpeta_respaldo <- if (dir.exists('/content/drive/MyDrive')) '/content/drive/MyDrive/Tarea_11_DADA2_respaldo' else 'resultados'
dir.create(carpeta_respaldo, recursive=TRUE, showWarnings=FALSE)
cat('Carpeta de respaldos:', carpeta_respaldo, '\n')


## 2. Selección clínica (común a ambas opciones)
Cargue `SraRunTable.csv` original. Esta sección comprueba diagnósticos y disponibilidad de FASTQ. Si no hay seis controles y seis Crohn verificados, se detiene: no se inventan grupos.


In [ ]:
# Validación de metadatos y selección de muestras (ambas opciones)
# Requiere SraRunTable.csv original descargado de NCBI.
archivos <- c('SraRunTable.csv', 'SraRunTable.txt')
archivo <- archivos[file.exists(archivos)][1]
if (is.na(archivo)) stop('Suba SraRunTable.csv original a Colab antes de continuar.')
metadata_original <- read.csv(archivo, check.names=FALSE, stringsAsFactors=FALSE)
if (ncol(metadata_original)<2) metadata_original <- read.delim(archivo, check.names=FALSE, stringsAsFactors=FALSE)
obligatorias <- c('Run','BioSample','LibraryLayout')
if (!all(obligatorias %in% names(metadata_original))) stop('Faltan columnas Run, BioSample o LibraryLayout.')
if (!any(c('Host_disease','gastrointest_disord') %in% names(metadata_original))) stop('No hay diagnósticos clínicos verificables.')
extraer <- function(col) if (col %in% names(metadata_original)) as.character(metadata_original[[col]]) else rep(NA_character_,nrow(metadata_original))
raw <- data.frame(Run=extraer('Run'), BioSample=extraer('BioSample'), LibraryLayout=extraer('LibraryLayout'), Host_disease=extraer('Host_disease'), gastrointest_disord=extraer('gastrointest_disord'), stringsAsFactors=FALSE)
cat('Diagnósticos Host_disease:\n'); print(table(raw$Host_disease,useNA='ifany'))
cat('Diagnósticos gastrointest_disord:\n'); print(table(raw$gastrointest_disord,useNA='ifany'))
cat('LibraryLayout:\n'); print(table(raw$LibraryLayout,useNA='ifany'))
normaliza <- function(x) {
  y <- tolower(trimws(as.character(x)))
  y <- gsub('[_-]', ' ', y)
  z <- rep(NA_character_, length(y))
  z[y %in% c('control','healthy','healthy control','healthy controls','hc','non ibd','no disease')] <- 'Control'
  z[y %in% c('crohn',"crohn's disease",'crohns disease','crohn disease','cd')] <- 'Crohn'
  z
}
h <- normaliza(raw$Host_disease); g <- normaliza(raw$gastrointest_disord)
if (any(!is.na(h) & !is.na(g) & h != g)) stop('Diagnósticos contradictorios: revisar tabla original.')
raw$Grupo <- ifelse(!is.na(h),h,g)
# Mostrar casos sin clasificar: nunca asignar diagnósticos artificiales
cat('Filas sin grupo reconocido:',sum(is.na(raw$Grupo)),'\n')
validas <- raw[!is.na(raw$Grupo) & toupper(trimws(raw$LibraryLayout))=='PAIRED' & !is.na(raw$BioSample) & nzchar(raw$BioSample), ,drop=FALSE]
# Excluir muestras que tengan más de una etiqueta clínica entre corridas
por_bio <- split(validas$Grupo, validas$BioSample)
if (any(vapply(por_bio,function(x) length(unique(x))>1,logical(1)))) stop('BioSample con grupos inconsistentes.')
validas <- validas[!duplicated(validas$BioSample),,drop=FALSE]
cat('Muestras independientes elegibles por grupo:\n'); print(table(validas$Grupo))
if (!all(c('Control','Crohn') %in% validas$Grupo) || any(table(factor(validas$Grupo,levels=c('Control','Crohn')))<6)) stop('No hay seis muestras por grupo: revise las etiquetas originales mostradas arriba.')
# Consultar ENA antes de seleccionar para garantizar pares FASTQ disponibles
url <- paste0('https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJNA450340&result=read_run&fields=run_accession,sample_accession,secondary_sample_accession,library_layout,fastq_ftp&format=tsv')
info_all <- read.delim(url, sep='\t', check.names=FALSE, stringsAsFactors=FALSE)
info_all <- info_all[!is.na(info_all$fastq_ftp) & grepl(';',info_all$fastq_ftp,fixed=TRUE) & toupper(info_all$library_layout)=='PAIRED',,drop=FALSE]
validas <- validas[validas$Run %in% info_all$run_accession,,drop=FALSE]
cat('Con FASTQ R1/R2 disponibles:\n'); print(table(validas$Grupo))
if (any(table(factor(validas$Grupo,levels=c('Control','Crohn')))<6)) stop('No hay seis muestras por grupo con FASTQ paired-end disponibles.')
set.seed(12)
seleccion <- do.call(rbind,lapply(c('Control','Crohn'),function(gr) {x<-validas[validas$Grupo==gr,,drop=FALSE];x[sample(seq_len(nrow(x)),6),,drop=FALSE]}))
rownames(seleccion)<-NULL
info <- info_all[match(seleccion$Run,info_all$run_accession),,drop=FALSE]
stopifnot(nrow(info)==12,!anyNA(info$run_accession),length(unique(seleccion$BioSample))==12)
# Validación de correspondencia robusta entre ENA y NCBI SRA
coincide_biosample <- as.character(info$sample_accession) == as.character(seleccion$BioSample) |
                      as.character(info$secondary_sample_accession) == as.character(seleccion$BioSample)
if (!all(coincide_biosample)) stop('BioSample de NCBI no coincide con ENA: revisar correspondencia.')
cat('SELECCIÓN FINAL (12 muestras):\n');print(seleccion[,c('Run','BioSample','Host_disease','gastrointest_disord','Grupo')]);print(table(seleccion$Grupo))
write.csv(seleccion,'metadata_clinicos_validados.csv',row.names=FALSE)
dir.create('datos',showWarnings=FALSE);dir.create('filtrados',showWarnings=FALSE)

# Guardar selección en Drive si está montado (ver celda de preparación)
saveRDS(seleccion, file.path(carpeta_respaldo, 'seleccion.rds'))
write.csv(seleccion, file.path(carpeta_respaldo, 'metadata_clinicos_validados.csv'), row.names=FALSE)



## 3. Obtener los FASTQ — ejecutar SOLO UNA opción
**Opción A:** descargar automáticamente mediante R. **Opción B:** subir los 24 FASTQ originales desde el panel Archivos de Colab a la carpeta `datos` (o copiarlos allí desde Drive). Si ya descargó los FASTQ, no ejecute A.


### Opción A · Descarga por código


In [ ]:
options(timeout=3600)
for (i in seq_len(nrow(info))) {
  links <- strsplit(info$fastq_ftp[i],';',fixed=TRUE)[[1]]
  if(length(links)!=2) stop('Corrida sin exactamente dos archivos FASTQ: ',info$run_accession[i])
  for (j in 1:2) {
    destino <- file.path('datos',paste0(info$run_accession[i],'_R',j,'.fastq.gz'))
    link <- if(grepl('^https?://',links[j])) links[j] else paste0('https://',links[j])
    if (!file.exists(destino) || file.size(destino)==0) {
      cat('Descargando ',basename(destino),'\n')
      download.file(link,destino,mode='wb')
    } else cat('Ya existe: ',basename(destino),'\n')
    if (!file.exists(destino) || file.size(destino)==0) stop('Archivo ausente o vacío: ',destino)

  }
}
fnFs <- sort(list.files('datos',pattern='_R1[.]fastq[.]gz$',full.names=TRUE))
fnRs <- sort(list.files('datos',pattern='_R2[.]fastq[.]gz$',full.names=TRUE))
sample.names <- sub('_R1[.]fastq[.]gz$','',basename(fnFs))
stopifnot(length(fnFs)==12,length(fnRs)==12,identical(sample.names,sub('_R2[.]fastq[.]gz$','',basename(fnRs))))
cat('Descarga comprobada: ',length(fnFs),' R1 y ',length(fnRs),' R2\n')
names(fnFs) <- names(fnRs) <- sample.names



### Opción B · Carga directa de FASTQ
En Colab, abra **Archivos**, cree/abra `datos` y cargue los 24 archivos `.fastq.gz` (12 R1 y 12 R2). Alternativamente, cópielos desde Drive a `datos`. Luego ejecute la siguiente celda. La carga directa requiere que los nombres de los archivos coincidan con las corridas validadas.


In [ ]:
# Ejecutar SOLO para la opción B (después de subir o copiar los archivos)
sample.names <- as.character(seleccion$Run)
fnFs <- file.path('datos', paste0(sample.names,'_R1.fastq.gz'))
fnRs <- file.path('datos', paste0(sample.names,'_R2.fastq.gz'))
names(fnFs) <- names(fnRs) <- sample.names
stopifnot(all(file.exists(fnFs)),all(file.exists(fnRs)))
cat('Se encontraron',length(fnFs),'archivos R1 y',length(fnRs),'archivos R2\n')



### Comprobación común: emparejamiento antes de DADA2
El mismo número de lecturas es necesario pero no suficiente para garantizar correspondencia de pares. Si hay diferencias, **no ejecute el filtrado**: revise la descarga o recupere los pares correctamente.


In [ ]:
stopifnot(length(fnFs)==12,length(fnRs)==12,all(file.exists(fnFs)),all(file.exists(fnRs)))
contar <- function(f) sum(ShortRead::countFastq(dirname(f),pattern=paste0('^',basename(f),'$'))$records)
nF <- vapply(fnFs,contar,numeric(1)); nR <- vapply(fnRs,contar,numeric(1))
revision <- data.frame(Muestra=sample.names,R1=unname(nF),R2=unname(nR),Coinciden=unname(nF==nR))
print(revision)
if (!all(revision$Coinciden)) stop('Hay R1/R2 con diferente cantidad de lecturas. Revisar antes de filtrar.')
# Nota: si la fuente contiene lecturas huérfanas, también deben comprobarse IDs/orden.



## 3. Calidad

In [ ]:
# Escribe aquí el código de esta etapa.


**Pregunta 3.** ¿Qué diferencias hay entre la calidad de R1 y R2?

**Tu respuesta:** ______________________________

## 4. Filtrado

In [ ]:
# Escribe aquí el código de esta etapa.


In [ ]:
# Guardar el resultado de esta etapa
saveRDS(out, file.path(carpeta_respaldo,'01_filtrado_out.rds'))
cat('Respaldo del filtrado guardado\n')


Respaldo del filtrado guardado


**Pregunta 4.** ¿Por qué se utilizan truncLen y maxEE? ¿Qué verificarías para garantizar solapamiento?

**Tu respuesta:** ______________________________

## 5. Errores y desreplicación

In [ ]:
# Escribe aquí el código de esta etapa.


In [ ]:
# Guardar el resultado de esta etapa
saveRDS(list(errF=errF,errR=errR,derepFs=derepFs,derepRs=derepRs), file.path(carpeta_respaldo,'02_errores_desrep.rds'))


**Pregunta 5.** ¿Para qué sirven learnErrors() y derepFastq()?

**Tu respuesta:** ______________________________

## 6. Inferencia y ensamblaje

In [ ]:
# Escribe aquí el código de esta etapa.


In [ ]:
# Guardar el resultado de esta etapa
saveRDS(list(dadaFs=dadaFs,dadaRs=dadaRs,mergers=mergers,seqtab=seqtab),file.path(carpeta_respaldo,'03_inferencia_ensamblaje.rds'))


**Pregunta 6.** ¿Qué hace mergePairs() y por qué pueden perderse lecturas?

**Tu respuesta:** ______________________________

## 7. Quimeras y seguimiento

In [ ]:
# Escribe aquí el código de esta etapa.


In [ ]:
# Guardar el resultado de esta etapa
saveRDS(list(seqtab.nochim=seqtab.nochim,track=track),file.path(carpeta_respaldo,'04_quimeras_seguimiento.rds'))

**Pregunta 8.** ¿Qué proporción de lecturas se conserva en cada muestra?

**Tu respuesta:** ______________________________

## 8. Taxonomía SILVA

In [ ]:
# 1. Crear carpeta
dir.create("referencias", showWarnings = FALSE)

# 2. Definir archivos SILVA 138.1
ref <- "referencias/silva_nr99_v138.1_train_set.fa.gz"

ref_species <- "referencias/silva_species_assignment_v138.1.fa.gz"

# 3. Enlaces oficiales
url_ref <- paste0(
  "https://zenodo.org/records/4587955/files/",
  "silva_nr99_v138.1_train_set.fa.gz?download=1"
)

url_species <- paste0(
  "https://zenodo.org/records/4587955/files/",
  "silva_species_assignment_v138.1.fa.gz?download=1"
)

# 4. Descargar si todavía no existen
options(timeout = 1800)

descargar_ref <- function(url, destino) {
  if (!file.exists(destino) || file.size(destino) == 0) {
    download.file(url, destino, mode = "wb", quiet = FALSE)
  }
  stopifnot(file.exists(destino), file.size(destino) > 0)
}

descargar_ref(url_ref, ref)
descargar_ref(url_species, ref_species)

# 5. Clasificación taxonómica
taxa <- assignTaxonomy(
  seqtab.nochim,
  refFasta = ref,
  multithread = FALSE
)

# 6. Asignación de especies
taxa <- addSpecies(
  taxa,
  refFasta = ref_species,
  verbose = TRUE
)

# 7. Revisar resultados
head(taxa)

In [ ]:
# Guardar el resultado de esta etapa
saveRDS(taxa,file.path(carpeta_respaldo,'05_taxonomia.rds'))

**Pregunta 9.** ¿Qué limitaciones tiene la asignación taxonómica con 16S V4?

**Tu respuesta:** ______________________________

## 9. Metadatos y phyloseq

In [ ]:
dir.create('resultados',showWarnings=FALSE)
metadata <- seleccion[match(rownames(seqtab.nochim), seleccion$Run),,drop=FALSE]
if(anyNA(metadata$Run)) stop('Hay muestras procesadas que no aparecen en los metadatos seleccionados.')
metadata$SampleID <- metadata$Run
rownames(metadata) <- metadata$SampleID
stopifnot(identical(rownames(seqtab.nochim),rownames(metadata)),
          !anyNA(metadata$Grupo),
          all(table(factor(metadata$Grupo,levels=c('Control','Crohn')))==6),
          identical(rownames(taxa),colnames(seqtab.nochim)))
ps <- phyloseq(otu_table(seqtab.nochim,taxa_are_rows=FALSE),tax_table(taxa),sample_data(metadata))
print(ps);print(table(sample_data(ps)$Grupo))
print(c(muestras=nsamples(ps),ASV=ntaxa(ps)))
if(nsamples(ps)!=12) stop('El objeto phyloseq no contiene 12 muestras.')
saveRDS(ps,'resultados/Phyloseq_practico.rds')
write.csv(metadata,'resultados/metadata_clinicos.csv',row.names=FALSE)
write.csv(track,'resultados/seguimiento_lecturas.csv')


In [ ]:
saveRDS(ps,file.path(carpeta_respaldo,'06_phyloseq_final.rds'))
write.csv(metadata,file.path(carpeta_respaldo,'metadata_clinicos.csv'),row.names=FALSE)
write.csv(track,file.path(carpeta_respaldo,'seguimiento_lecturas.csv'))


**Pregunta 10.** ¿Cuántas muestras contiene el objeto `phyloseq` y cómo se distribuyen por grupo?

**Tu respuesta:** ______________________________


### Preparación para la Tarea 12
El objeto se guarda como `ps` en `resultados/Phyloseq_practico.rds`. En la próxima tarea recuperaremos este archivo con `ps <- readRDS("resultados/Phyloseq_practico.rds")`.


## Recuperar resultados en otra sesión
Monte Drive y configure `carpeta_respaldo` como en la primera celda. Ejecute **solo** la recuperación del punto que necesite; los archivos RDS preservan los objetos, pero los FASTQ filtrados deben seguir disponibles para etapas que los requieren.


In [ ]:
# Ejemplos de recuperación: descomente solo los bloques que necesite.
# seleccion <- readRDS(file.path(carpeta_respaldo,'seleccion.rds'))
# out <- readRDS(file.path(carpeta_respaldo,'01_filtrado_out.rds'))
# etapa2 <- readRDS(file.path(carpeta_respaldo,'02_errores_desrep.rds')); list2env(etapa2,.GlobalEnv)
# etapa3 <- readRDS(file.path(carpeta_respaldo,'03_inferencia_ensamblaje.rds')); list2env(etapa3,.GlobalEnv)
# etapa4 <- readRDS(file.path(carpeta_respaldo,'04_quimeras_seguimiento.rds')); list2env(etapa4,.GlobalEnv)
# taxa <- readRDS(file.path(carpeta_respaldo,'05_taxonomia.rds'))
# ps <- readRDS(file.path(carpeta_respaldo,'06_phyloseq_final.rds'))



## 10. Descargar resultados y respaldos


## 10. Descargar resultados

In [ ]:
# Comprimir resultados locales. Los respaldos en Drive permanecen en carpeta_respaldo.
archivos_zip <- list.files('resultados',full.names=TRUE)
if(length(archivos_zip)) utils::zip('resultados_semana12.zip',files=archivos_zip)
file.exists('resultados_semana12.zip')


## Preguntas finales
1. ¿Cuál es el objetivo del estudio y qué comparación clínica realizaremos?
2. ¿Qué proporción de lecturas se retiene después de filtrar y eliminar quimeras?
3. ¿Cuántos ASV y muestras contiene `phyloseq`?
4. ¿Cuántas muestras pertenecen a Control y Crohn?
5. ¿Por qué no basta con los nombres de FASTQ para conocer el diagnóstico?

**Prueba opcional de visualización para la semana 13:**
```r
plot_richness(ps, x="Grupo", measures="Shannon")
```


## Evaluación sugerida (20 puntos)
- Descarga y correspondencia R1/R2: 3
- Calidad y filtrado: 4
- DADA2 y ensamblaje: 5
- Quimeras y taxonomía: 3
- `phyloseq` y guardado: 3
- Interpretación y límites: 2

**Estado de validación:** notebook revisado estructuralmente, no ejecutado extremo a extremo en Colab. Las etiquetas clínicas por corrida requieren comprobación documental antes de una comparación de pacientes.